In [ ]:
# House Price Prediction - Exploratory Data Analysis
# Cleans the dataset and saves it for model training

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# Load dataset
df = pd.read_csv('./Data/House_Data.csv')
df.head()

In [ ]:
# Basic dataset info
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing[missing > 0]

In [ ]:
# Extract bedroom count from strings like "2 BHK" or "4 Bedroom"
df['bedrooms'] = df['size'].str.extract(r'(\d+)').astype(float)

In [ ]:
# Some total_sqft values are ranges like "2100 - 2850", take the average
def parse_sqft(value):
    if isinstance(value, str):
        value = value.strip()
        if '-' in value:
            low, high = value.split('-')
            return (float(low.strip()) + float(high.strip())) / 2
    try:
        return float(value)
    except (ValueError, TypeError):
        return np.nan

df['total_sqft'] = df['total_sqft'].apply(parse_sqft)

In [ ]:
# Fill missing bath and balcony with median values
df['bath'] = df['bath'].fillna(df['bath'].median())
df['balcony'] = df['balcony'].fillna(df['balcony'].median())

In [ ]:
# Keep only features needed for modeling
feature_cols = ['square_footage', 'bedrooms', 'bathrooms', 'price']

df_clean = df.rename(columns={
    'total_sqft': 'square_footage',
    'bath': 'bathrooms'
})

model_data = df_clean[feature_cols].dropna()

print(f'Rows before cleaning: {len(df)}')
print(f'Rows after dropping missing values: {len(model_data)}')
model_data.head()

In [ ]:
model_data.describe()

In [ ]:
# Save cleaned data for model training notebook
output_path = './Data/house_data_cleaned.csv'
model_data.to_csv(output_path, index=False)
print(f'Saved cleaned data to {output_path}')

In [ ]:
# Price and square footage distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(model_data['price'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price (lakhs)')
axes[0].set_ylabel('Count')

axes[1].hist(model_data['square_footage'], bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Square Footage Distribution')
axes[1].set_xlabel('Square Footage')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Price vs square footage and bedrooms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(model_data['square_footage'], model_data['price'], alpha=0.4, s=15)
axes[0].set_title('Price vs Square Footage')
axes[0].set_xlabel('Square Footage')
axes[0].set_ylabel('Price')

axes[1].scatter(model_data['bedrooms'], model_data['price'], alpha=0.4, s=15, color='coral')
axes[1].set_title('Price vs Bedrooms')
axes[1].set_xlabel('Bedrooms')
axes[1].set_ylabel('Price')

plt.tight_layout()
plt.show()

In [ ]:
# Price vs bathrooms
plt.figure(figsize=(8, 5))
plt.scatter(model_data['bathrooms'], model_data['price'], alpha=0.4, s=15, color='purple')
plt.title('Price vs Bathrooms')
plt.xlabel('Bathrooms')
plt.ylabel('Price')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Feature correlation
corr = model_data.corr()
print(corr)

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation')
plt.show()